## 📐💧 Precipitation Properties and Geometry 💧📐

From ORCESTRA/PICCOLO SEA-POL radar observations, calculate precipitation properties: area mean precipitation rate (P), conditional mean precipitation rate (I), and precipitation area fraction ($f_A$). Identify precipitating entities and calculate basic geometries: entity number (N) and average entity area ($\overline{A}$). (Code for the computation of edge-to-edge distance between all possible entity pairs ($\overline{D}$) can be found in the organization metrics notebook in this repository.) 

SEA-POL data is available through IPFS at http://127.0.0.1:8080/ipns/latest.orcestra-campaign.org/products/METEOR/SEA-POL/. 

For more on data access with IPFS, see https://orcestra-campaign.org/data.html. 

## ⬇️ Import relevant libraries, access SEA-POL radar data, and process variables. 

In [1]:
import numpy as np
import xarray as xr
import netCDF4 as nc
import pandas as pd
from datetime import datetime
from scipy.stats import norm
from scipy.ndimage import gaussian_filter, label, generate_binary_structure 

In [2]:
# Access the PICCOLO_level4_rainrate_2D.zarr dataset at the following URL:
url = "http://127.0.0.1:8080/ipns/latest.orcestra-campaign.org/products/METEOR/SEA-POL/PICCOLO_level4_rainrate_2D.zarr/"

# Open the 2D dataset using xarray with the zarr engine. 
ds = xr.open_dataset(url, engine="zarr")
vars_list = list(ds.variables.keys())

# Print a list of available variables in the dataset.
print(vars_list)

['DBZ', 'HID', 'RAINRATE', 'X', 'Y', 'elevation_angle', 'grid_mapping', 'heading', 'latitude', 'longitude', 'start_time', 'stop_time', 'time']


In [3]:
# Name relevant variables and read them in. 
dbz = ds.variables['DBZ'][:]
print('done with dbz') # Print 'done' statements to track loading progess. 

rainrate = ds.variables['RAINRATE'][:]
print('done with rainrate')

x = ds.variables['X'][:]
print('done with x')

y = ds.variables['Y'][:]
print('done with y')

latitude = ds.variables['latitude'][:]
print('done with latitude')

longitude = ds.variables['longitude'][:]
print('done with longitude')

start_time = ds.variables['start_time'][:]
print('done with start_time')

stop_time = ds.variables['stop_time'][:]
print('done with stop_time')

time = ds.variables['time'][:]
print('done with time')

done with dbz
done with rainrate
done with x
done with y
done with latitude
done with longitude
done with start_time
done with stop_time
done with time


In [4]:
# Spatially subset radar data to within a specified radius from SEA-POL. 

# Re-name rainrate and reflectivity arrays for easier use. 
dbz = dbz[:, :, :]
rr = rainrate[:, :, :]

# Create 2D meshgrid of x and y. 
X, Y = np.meshgrid(x, y)

# Calculate the distance from the center for each grid point. 
center_x = float((x.max() + x.min()) / 2)
center_y = float((y.max() + y.min()) / 2)
distance_from_center = np.sqrt((X - center_x)**2 + (Y - center_y)**2)

# Create a mask for points within desired radius. 
radius_mask = distance_from_center <= 120000 # 120 km radius in meters


# Apply the mask to all rainrate arrays. 
rr_120km = np.where(radius_mask, rr, np.nan)

# Ensure that NaNs, -32768, and masked values ('--') in the original arrays are preserved in the masked arrays. 
rr_120km = np.where(np.isnan(rr) | (rr == -32768), np.nan, rr_120km)



# Apply the mask to all dbz arrays. 
dbz_120km = np.where(radius_mask, dbz, np.nan)

# Ensure that NaNs, -32768, and masked values ('--') in the original arrays are preserved in the masked arrays
dbz_120km = np.where(np.isnan(dbz) | (dbz == -32768), np.nan, dbz_120km)

## ⛈️ Compute and store metrics that describe precipitation properties (P, I, $f_A$).
This code block uses the 'rr_120km' list of 2D rainrate arrays subset to a 120 km radius from the SEA-POL radar as an example. The same logic is used for different radial distances from the radar or strength of radar echoes. The files with the computed metrics for each radius and strength bin can be found in the Files folder of the repository. 

In [6]:
# Create a set of lists to store the results from each radar scene. 
precip_list = []
intensity_list = []
fa_list = []


# Calculate P, I, fA for each radar scan over a given set of times.  
for t in range(len(rr_120km)): # loop through each scan (time step) in the masked rainrate arrays

    # Process each scan one at a time in the for loop. 
    indiv_scan_rainrate = rr_120km[t,:,:]

    # Area-mean precipitation rate (P, units of mm/hr).
    # Average rain rate over all scanned pixels (including non-raining pixels). 
    indiv_scan_rainrate = np.where(indiv_scan_rainrate < -30000, np.nan, indiv_scan_rainrate) # make sure that all non-scanned pixles are NaNs
    indiv_scan_rainrate = np.where((indiv_scan_rainrate == -9999), 0, indiv_scan_rainrate) # set all pixels that were scanned but non raining to 0 (Extra redundant sanity check!) 
    indiv_scan_rainrate = np.where((indiv_scan_rainrate <= 0), 0, indiv_scan_rainrate) # take the mean of all pixels that were scanned no matter the rain rate
    precip_val = np.nanmean(indiv_scan_rainrate)
    precip_list.append(precip_val) # append to list

    # Conditional-mean precipitation rate or rainrate intensity (I, units of mm/hr). 
    # Average rain rate over only raining pixels (rain rate > 0).
    indiv_scan_rainrate_all_nan = np.where(indiv_scan_rainrate <= 0 , np.nan, indiv_scan_rainrate) # set all non-raining pixels to NaN
    intensity_val = np.nanmean(indiv_scan_rainrate_all_nan) # take the mean of all raining pixels 
    intensity_list.append(intensity_val) # append to list

    # Fractional area of precipitation (fA, unitless).
    # Count the number of pixels with rain rate > 0. 
    num_raining_pixels = np.sum(indiv_scan_rainrate > 0)
    total_valid_pixels = np.sum(indiv_scan_rainrate >= -10000) # captures all cells
    if total_valid_pixels > 0:
        frac = num_raining_pixels / total_valid_pixels # fA is ratio of raining pixels to total scanned pixels
    else:
        frac = np.nan

    fa_list.append(frac)


## 📏 Identify precipitating entities. Compute and store metrics that describe basic precipitation entity geometries. (N, $\overline{A}$).
This code block uses the 'dbz_120km' list of 2D reflectivity arrays subset to a 120 km radius from the SEA-POL radar as an example. The same logic is used for different radial distances from the radar or strength of radar echoes. The files with the computed metrics for each radius and strength bin can be found in the Files folder of the repository. 

In [7]:
# Create a set of lists to store the results from each radar scene. 
n_list = []
abar_list = []

# Identify precipitating entities and compute geometries for each radar scan. 
for t in range(len(dbz_120km)): # loop through each scan (time step) in the masked reflectivity arrays

    # Process each scan one at a time in the for loop. 
    dbz_array = dbz_120km[t]

    if not np.all(np.isnan(dbz_array)):

        # Create a binary mask where pixels with an echo (larger than -30 dBZ) are given a 1 and all other pixels are 0. 
        rain_mask = np.where(np.isnan(dbz_array), 0, np.where(dbz_array > -30, 1, 0)) 

        # Set up an 8-connectivity structure for identifying contiguous echo regions.
        s = generate_binary_structure(2, 2)

        # Use scipy's label function to identify contiguous regions in the binary mask. 
        # 'entity_mask' will have unique numerical labels for each entity. 
        # 'nentities' is the total number of entities found
        entity_mask, nentities = label(rain_mask, structure=s)

        # Store the results. 
        n_list.append(nentities)

        # Calculate the area of each entity by counting the number of pixels in each labeled region (1 pixel = 1 km^2). 
        cluster_counts = [] # stores area of each individual entity in the scene
        if nentities > 0:
            # Loop through each label number.  
            for cluster_id in range(1, nentities + 1): # count starts at 1. white space is 0.
                count = np.sum(entity_mask == cluster_id)
                cluster_counts.append(count)

        if cluster_counts:
            mean_count = np.mean(cluster_counts) # mean area of entities in the scene
        else:
            mean_count = np.nan

        # Store the results. 
        abar_list.append(mean_count) 
        
    else:
        # Return NaNs if the array has no echoes (severe clear).
        n_list.append(np.nan)
        abar_list.append(np.nan)